In [ ]:
# 우리가 원하는 아웃풋의 형식을 만드는 법
# 2를 보면 실행 결과가 AI 메시지임 근데 사용자가 원하는건 Paris
# Paris로 다음 작업을 하거나 기록을 하고 싶을 때 장황한 AIMessage에서 원하는 것만 뺴오기

from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3.2:1b")

In [ ]:
from langchain_core.prompts  import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt_template = PromptTemplate(
    template="What is the capital of {country}?",
    input_variables=["country"]
)

prompt = prompt_template.invoke({"country": "France"})
ai_response = llm.invoke(prompt)

output_parser = StrOutputParser()
answer = output_parser.invoke(llm.invoke(prompt))

# 'The capital of France is Paris.'


In [ ]:
ai_response
# 이건 AIMessage라는 클래스

AIMessage(content='The capital of France is Paris.', additional_kwargs={}, response_metadata={'model': 'llama3.2:1b', 'created_at': '2026-02-22T08:36:45.190434Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1263560292, 'load_duration': 110559375, 'prompt_eval_count': 32, 'prompt_eval_duration': 957157834, 'eval_count': 8, 'eval_duration': 183577085, 'logprobs': None, 'model_name': 'llama3.2:1b', 'model_provider': 'ollama'}, id='lc_run--019c847e-6d0a-7ce0-be1c-a28b63ac1892-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 32, 'output_tokens': 8, 'total_tokens': 40})

In [ ]:
answer
# 이건 str이라서 그냥 문자열로 나옴

# 이렇게 하면 더 좋음
# 예를들어 python 서버에서 돌려서 javascript 프론트로 전달
# 그럼 해당 클래스가 없으니까 string으로 줘야 됨

'The capital of France is Paris.'

In [12]:
# 여기서 또 Paris만 원하는 경우
prompt_template = PromptTemplate(
    template="What is the capital of {country}? Return only the capital name.",
    input_variables=["country"]
)

prompt = prompt_template.invoke({"country": "France"})

output_parser = StrOutputParser()
answer = output_parser.invoke(llm.invoke(prompt))
print(answer)


Paris


In [16]:
# string말고 JSON 형식으로 줄 수 있음
from langchain_core.output_parsers import JsonOutputParser

country_detail_prompt = PromptTemplate(
    template="""
    What is the capital of {country}?
    - capital: <capital>
    - population: <population>
    - area: <area>
    - language: <language>

    Return the result in JSON format.
    """,
    input_variables=["country"]
)
country_detail_prompt.invoke({"country": "France"})

output_parser = JsonOutputParser()
json_ai_msg = llm.invoke(country_detail_prompt.invoke({"country": "France"}))
# output_parser.invoke(json_ai_msg)

# answer = output_parser.invoke(llm.invoke(prompt))

# JSONDecodeError

In [ ]:
json_ai_msg
# 이것도 타입이 String이라서 파싱을 해야 됨
# 이렇게 하면 파싱이 안 됨
# output_parser.invoke(json_ai_msg)

# 이렇게 하면 파싱이 됨
output_parser.invoke(json_ai_msg)

# 그리고 이거 형식이 오락가락함
# 있다는 전제로 replace를 하면 오류 가능성이 커짐 -> 실질적으로 사용하지 않음

# 대신 pydantic으로 모델 설정 가능


{'capital': '<capital>',
 'population': '<population>',
 'area': '<area>',
 'language': '<language>'}

In [20]:
from pydantic import BaseModel, Field

class CountryDetail(BaseModel):
    capital: str = Field(description="The capital of the country")
    population: int = Field(description="The population of the country")
    area: float = Field(description="The area of the country in square kilometers")
    language: str = Field(description="The official language of the country")   

structured_llm = llm.with_structured_output(CountryDetail)
response = structured_llm.invoke(country_detail_prompt.invoke({"country": "France"}))


In [ ]:
response
# 위에서 선언한 파이덴틱 모델로 나옴

CountryDetail(capital='<capital>', population=65, area=6436.0, language='French')

In [22]:
response.capital


'<capital>'

In [23]:
response.population

65

In [ ]:
response.model_dump()
# JSON 형식으로 나옴
# 이렇게 사용해야 우리가 원하는 형태를 일정하게 받을 수 있음

{'capital': '<capital>',
 'population': 65,
 'area': 6436.0,
 'language': 'French'}

In [26]:
response.model_dump()['area']

6436.0